In [ ]:
# UMAP overlay: TOX21 training set (fit) and ZINC15 (projected)
# If packages are missing we try to install them (works in most notebook environments).
import importlib, subprocess, sys


# Ensure required packages
#ensure("rdkit-pypi", "rdkit")

import os
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
import umap
import matplotlib.pyplot as plt

# --- Load TOX21 training SMILES ---
# Adjust path if your tox21 file is elsewhere. The repository contains `data/tox21.csv.gz`.
tox21_path = "data/tox21.csv.gz"
if not os.path.exists(tox21_path):
    raise FileNotFoundError(f"Could not find TOX21 file at {tox21_path}. Update `tox21_path` to point to your file.")

# Read CSV (supports gzip)
df = pd.read_csv(tox21_path, compression='gzip' if tox21_path.endswith('.gz') else None)
# Try to detect SMILES column
smiles_col = None
for c in ['smiles', 'SMILES', 'canonical_smiles', 'smile']:
    if c in df.columns:
        smiles_col = c
        break
if smiles_col is None:
    for c in df.columns:
        if 'smile' in c.lower():
            smiles_col = c
            break
if smiles_col is None:
    raise ValueError(f"Couldn't detect a SMILES column in {tox21_path}. Columns: {df.columns.tolist()}")

tox_smiles = df[smiles_col].dropna().astype(str).tolist()
print(f"Loaded {len(tox_smiles)} TOX21 molecules from column '{smiles_col}'")

# --- Load ZINC15 SMILES ---
# Set `zinc_path` to a local SMILES file (one SMILES per line or SMILES\tID). If you don't have it locally, download a subset from ZINC15.
zinc_path = "data/zinc15.smi"
if not os.path.exists(zinc_path):
    raise FileNotFoundError(f"ZINC SMILES file not found at {zinc_path}. Please provide a local file with one SMILES per line and set `zinc_path` accordingly.")

zinc_smiles = []
with open(zinc_path, 'r') as fh:
    for line in fh:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        zinc_smiles.append(parts[0])
print(f"Loaded {len(zinc_smiles)} ZINC SMILES")

# To avoid memory issues, subsample ZINC if it's enormous
max_zinc = 200000
if len(zinc_smiles) > max_zinc:
    import random
    zinc_smiles = random.sample(zinc_smiles, max_zinc)
    print(f"Subsampled ZINC to {len(zinc_smiles)} molecules for plotting")

# --- Convert SMILES to RDKit molecules and compute Morgan fingerprints ---
from tqdm.auto import tqdm

def mols_from_smiles(smiles_list):
    mols = []
    for s in smiles_list:
        m = Chem.MolFromSmiles(s)
        mols.append(m)
    return mols


def morgan_array(mols, n_bits=2048, radius=2):
    arr = np.zeros((len(mols), n_bits), dtype=np.uint8)
    for i, m in enumerate(mols):
        if m is None:
            continue
        fp = AllChem.GetMorganFingerprintAsBitVect(m, radius, nBits=n_bits)
        arr[i, :] = np.frombuffer(fp.ToBitString().encode('ascii'), dtype='S1')
        # The above line returns ascii bytes; convert to 0/1
        arr[i, :] = np.array([int(b) for b in fp.ToBitString()], dtype=np.uint8)
    return arr

print('Converting TOX21 SMILES to molecules...')
tox_mols = mols_from_smiles(tqdm(tox_smiles))
print('Converting ZINC SMILES to molecules...')
zinc_mols = mols_from_smiles(tqdm(zinc_smiles))

print('Computing Morgan fingerprints (binary)...')
tox_fp = morgan_array(tqdm(tox_mols))
zinc_fp = morgan_array(tqdm(zinc_mols))

# --- Fit UMAP on TOX21 and project ZINC ---
print('Fitting UMAP on TOX21 fingerprints...')
reducer = umap.UMAP(n_components=2, random_state=42, metric='jaccard')
tox_emb = reducer.fit_transform(tox_fp)
print('Projecting ZINC fingerprints into the same UMAP space...')
zinc_emb = reducer.transform(zinc_fp)

# --- Plot ---
import matplotlib.pyplot as plt
plt.figure(figsize=(8,8))
plt.scatter(zinc_emb[:,0], zinc_emb[:,1], s=2, color='gray', alpha=0.4, label='ZINC15')
plt.scatter(tox_emb[:,0], tox_emb[:,1], s=6, color='red', alpha=0.9, label='TOX21 (train)')
plt.legend(markerscale=4)
plt.title('UMAP: TOX21 (fit) + ZINC15 (projected)')
plt.xlabel('UMAP1')
plt.ylabel('UMAP2')
plt.tight_layout()
plt.show()


/Users/iamthomaspruyn/Documents/PhD/Projects/AI-for-Toxicology/AI-for-Toxicology/.venv/bin/python: No module named pip


CalledProcessError: Command '['/Users/iamthomaspruyn/Documents/PhD/Projects/AI-for-Toxicology/AI-for-Toxicology/.venv/bin/python', '-m', 'pip', 'install', 'umap-learn']' returned non-zero exit status 1.